# GRU-D H1 → Joint Six-M4 Baseline

Fresh single-P100 training from step 0 to step 4000. The two private Inputs are already bound by the published Notebook. This job trains and exports only; full teacher-forced and free-running evaluation are separate jobs.

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json

SCIENCE_SCHEMA = 'trauma_predict.grud_h1_joint_m4_p100_bundle.v1'
SCIENCE_REF = 'vanila111/trauma-predict-grud-h1-joint-m4-v2-bundle'
matches = []
for manifest_path in sorted(Path('/kaggle/input').rglob('grud_v2_bundle_manifest.json')):
    if manifest_path.is_symlink() or not manifest_path.is_file():
        continue
    payload = json.loads(manifest_path.read_text(encoding='utf-8'))
    if payload.get('schema') == SCIENCE_SCHEMA and payload.get('dataset_ref') == SCIENCE_REF:
        matches.append((manifest_path.parent.resolve(), payload))
if len(matches) != 1:
    raise RuntimeError(f'Expected one pre-bound GRU-D science Input, found {len(matches)}')
bundle, manifest = matches[0]
launcher_row = manifest.get('launcher')
if not isinstance(launcher_row, dict):
    raise TypeError('Science manifest lacks launcher lock')
launcher = bundle / str(launcher_row.get('path') or '')
digest = hashlib.sha256(launcher.read_bytes()).hexdigest() if launcher.is_file() else ''
if launcher.is_symlink() or launcher.stat().st_size != int(launcher_row.get('size_bytes', -1)) or digest != launcher_row.get('sha256'):
    raise ValueError('Mounted GRU-D launcher differs from the science manifest')
spec = importlib.util.spec_from_file_location('grud_v2_bootstrap', launcher)
bootstrap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bootstrap)
SCIENCE_STATE = bootstrap.bind_science_input()


In [ ]:
RUNTIME_ENVIRONMENT = bootstrap.install_p100_runtime()


In [ ]:
bootstrap.run_training(SCIENCE_STATE, RUNTIME_ENVIRONMENT)
